# Arabic Legal Contract Risk Analysis System
### Full Production Pipeline

1. Loads the saved trained model 
2. Accepts a PDF or image contract
3. Extracts all clauses using an LLM
4. Predicts risk, affected parties, and risk type for each clause
5. Generates an Arabic explanation for each risky clause
6. Returns a structured JSON response



---
## Cell 1 — Setup: Install Dependencies

In [ ]:


# pip install groq pymupdf pytesseract pillow scikit-learn -q
# apt-get install -y tesseract-ocr tesseract-ocr-ara -q



Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 44.5 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
The following NEW packages will be installed:
  tesseract-ocr-ara
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 645 kB of archives.
After this operation, 1,447 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-ara all 1:4.00~git30-7274cfa-1.1 [645 kB]
Fetched 645 kB in 0s (3,774 kB/s)
Selecting previously unselected package tesseract-ocr-ara.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ara_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ara (1:4.00~git30-7274cfa-1.1) ...
Setting up tesser

---
## Cell 2 — Set Your API Key (run once per session)

In [2]:
# Safe way to set your API key — never paste it directly in the code
from getpass import getpass
import os

os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key: ')
print('✅ API key set')

Enter your Groq API key: ··········
✅ API key set


---
## Cell 3 — Load Saved Model from Drive

In [ ]:
import pickle
import numpy as np

SAVE_DIR = 'arabic_legal_model'

with open(f'{SAVE_DIR}/ridge_model_best.pkl',  'rb') as f: ridge_model           = pickle.load(f)
with open(f'{SAVE_DIR}/ridge_thresholds.pkl',  'rb') as f: best_thresholds_ridge = pickle.load(f)
with open(f'{SAVE_DIR}/mlb.pkl',               'rb') as f: mlb                   = pickle.load(f)
with open(f'{SAVE_DIR}/le_risk_type.pkl',      'rb') as f: le_risk_type          = pickle.load(f)
with open(f'{SAVE_DIR}/risk_type_ar_map.pkl',  'rb') as f: RISK_TYPE_AR          = pickle.load(f)
with open(f'{SAVE_DIR}/y.pkl',                 'rb') as f: y                     = pickle.load(f)

print('✅ Model loaded from Drive')
print('Party classes:   ', mlb.classes_)
print('Risk type classes:', le_risk_type.classes_)
print('Thresholds:      ', best_thresholds_ridge)

✅ Model loaded from Drive
Party classes:    ['الطرف الأول' 'الطرف الثاني' 'جميع الأطراف']
Risk type classes: ['contractual' 'financial' 'legal' 'liability' 'none' 'operational']
Thresholds:       {'risk': np.float64(1.7763568394002505e-15), 'الطرف الأول': np.float64(-0.3999999999999986), 'الطرف الثاني': np.float64(0.10000000000000187), 'جميع الأطراف': np.float64(0.10000000000000187)}


---
## Cell 4 — Core Prediction Functions

In [4]:
def to_arabic_risk_type(english_label):
    """Translates English risk_type label to Arabic for display."""
    return RISK_TYPE_AR.get(english_label, english_label)


def predict_clause(clause, contract_type='عام', contract_subtype='عام'):
    """
    Runs the saved RidgeClassifier with tuned thresholds on a single Arabic clause.
    Returns risk, affected parties, and risk type in Arabic.
    """
    full_text     = f'[{contract_type}] [{contract_subtype}] {clause}'
    X_transformed = ridge_model.named_steps['tfidf'].transform([full_text])
    estimators    = ridge_model.named_steps['clf'].estimators_
    y_columns     = list(y.columns)

    raw_pred = []
    for i, col in enumerate(y_columns):
        if col == 'risk_type_label':
            raw_pred.append(estimators[i].predict(X_transformed)[0])
        else:
            score = estimators[i].decision_function(X_transformed)[0]
            t     = best_thresholds_ridge[col]
            raw_pred.append(1 if score >= t else 0)

    risk       = int(raw_pred[0])
    party_cols = list(mlb.classes_)
    parties    = [party_cols[i] for i, v in enumerate(raw_pred[1:1+len(party_cols)]) if v == 1]

    risk_type_idx = int(raw_pred[-1])
    risk_type_en  = le_risk_type.classes_[risk_type_idx]
    risk_type_ar  = to_arabic_risk_type(risk_type_en)

    return {
        'risk':         risk,
        'risk_parties': parties        if risk == 1 else [],
        'risk_type':    risk_type_ar   if risk == 1 else None,
        'risk_type_en': risk_type_en   if risk == 1 else None,
    }


print('✅ predict_clause() ready')

✅ predict_clause() ready


---
## Cell 5 — LLM Explainability Layer (Groq API)

In [5]:
import os
import json
from groq import Groq

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

EXPLANATION_SYSTEM_PROMPT = (
    'أنت مساعد قانوني متخصص في تبسيط البنود التعاقدية العربية لغير المختصين. '
    'مهمتك الوحيدة هي تحليل بند تعاقدي معين بناءً فقط على نص البند والتصنيفات المعطاة. '
    'لا تضف أي افتراضات قانونية غير مدعومة بالنص. '
    'لا تستشهد بقوانين أو مواد قانونية محددة لم تُذكر في النص. '
    'اكتب بلغة عربية فصحى واضحة ومفهومة لغير المتخصصين في القانون. '
    'أجب دائمًا بصيغة JSON صالحة فقط بدون أي نص إضافي.'
)

def explain_clause_risk(clause, parties, risk_type_ar):
    """
    Calls LLM and returns a structured dict with 3 separate fields:
      - explanation:   why this clause is risky (2-4 sentences)
      - risk_level:    'منخفض' | 'متوسط' | 'مرتفع'
      - recommendation: what to do before signing (1-2 sentences)
    """
    parties_str = '، '.join(parties) if parties else 'غير محدد'

    user_prompt = f"""
البند التعاقدي:
\"{clause}\"

تصنيف النظام:
- وجود خطر: نعم
- الأطراف المتأثرة: {parties_str}
- نوع الخطر: {risk_type_ar}

أرجع تحليلك بصيغة JSON فقط بالشكل التالي بالضبط:
{{
  "explanation": "شرح موجز (2-4 جمل) يوضح لماذا يُعتبر هذا البند خطرًا وما الآثار القانونية أو المالية أو التشغيلية المحتملة وأي طرف هو الأكثر تأثرًا",
  "risk_level": "منخفض أو متوسط أو مرتفع فقط — اختر واحدة بناءً على مدى خطورة البند",
  "recommendation": "جملة أو جملتان تخبر المستخدم بما يجب الانتباه له أو فعله قبل التوقيع"
}}

قواعد صارمة:
- risk_level يجب أن يكون إحدى هذه القيم الثلاث فقط: منخفض، متوسط، مرتفع
- لا تضف أي نص خارج JSON
- اعتمد فقط على نص البند والتصنيفات المذكورة
"""

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': EXPLANATION_SYSTEM_PROMPT},
            {'role': 'user',   'content': user_prompt}
        ],
        temperature=0.3,
        max_tokens=500,
        response_format={'type': 'json_object'},  # forces valid JSON output
    )

    raw = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw)
        return {
            'explanation':    parsed.get('explanation', ''),
            'risk_level':     parsed.get('risk_level', 'متوسط'),
            'recommendation': parsed.get('recommendation', ''),
        }
    except json.JSONDecodeError:
        # Fallback if JSON parsing fails — return safe defaults
        return {
            'explanation':    raw,
            'risk_level':     'متوسط',
            'recommendation': '',
        }

print('✅ explain_clause_risk() ready — returns explanation + risk_level + recommendation')

✅ explain_clause_risk() ready — returns explanation + risk_level + recommendation


---
## Cell 6 — Document Ingestion (PDF / Image → Text)

In [6]:
import fitz          # PyMuPDF
import pytesseract
from PIL import Image
import io
import re

def normalize_arabic(text):
    """Fix common Tesseract Arabic OCR artifacts (dropped hamza, alef variants)."""
    # Normalize all alef variants to bare alef
    text = re.sub('[إأآ]', 'ا', text)
    # Remove stray non-Arabic/non-punctuation control characters
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text


def extract_text_from_pdf(pdf_path):
    """
    Extracts text from PDF. Uses native text extraction for digital PDFs,
    falls back to OCR for scanned/image-based pages.
    """
    doc = fitz.open(pdf_path)
    full_text = []

    for page in doc:
        text = page.get_text().strip()
        if len(text) < 20:   # scanned page — use OCR
            pix       = page.get_pixmap(dpi=300)
            img_bytes = pix.tobytes('png')
            img       = Image.open(io.BytesIO(img_bytes))
            text      = pytesseract.image_to_string(img, lang='ara')
        full_text.append(text)

    doc.close()
    return normalize_arabic('\n'.join(full_text))


def extract_text_from_image(image_path):
    """Extracts Arabic text from a standalone image using Tesseract OCR."""
    img  = Image.open(image_path)
    text = pytesseract.image_to_string(img, lang='ara')
    return normalize_arabic(text)


def extract_text(file_path):
    """Auto-detects file type and routes to the correct extractor."""
    ext = file_path.lower().split('.')[-1]
    if ext == 'pdf':
        return extract_text_from_pdf(file_path)
    elif ext in ('png', 'jpg', 'jpeg', 'tiff', 'bmp'):
        return extract_text_from_image(file_path)
    else:
        raise ValueError(f'Unsupported file type: .{ext}')


print('✅ Document extraction functions ready')

✅ Document extraction functions ready


---
## Cell 7 — LLM Clause Extractor

In [7]:
CLAUSE_EXTRACTION_PROMPT = """أنت أداة استخراج بنود قانونية دقيقة جدًا. مهمتك الوحيدة هي تقسيم نص عقد عربي إلى بنوده وأحكامه المنفصلة.

قواعد صارمة يجب الالتزام بها:
1. اقرأ النص الكامل للعقد المُعطى.
2. حدد وافصل كل بند أو مادة أو حكم أو التزام أو شرط أو حق أو مسؤولية أو جزاء أو نص تعاقدي مستقل.
3. تعامل مع جميع أنماط الترقيم والتنسيق التالية، حتى لو وردت مجتمعة في نفس المستند:
   - "البند الأول"، "البند الثاني"، إلخ
   - "المادة الأولى"، "المادة الثانية"، إلخ
   - ترقيم بالأرقام (1، 2، 3 أو 1-، 2-)
   - نقاط (bullet points)
   - فقرات بدون ترقيم واضح أو تنسيق غير متسق
4. احتفظ بالنص العربي الأصلي تمامًا كما ورد دون أي تغيير.
5. ممنوع تلخيص النص أو إعادة الصياغة أو تصحيح الأخطاء أو حذف أو إضافة أي كلمة.
6. أرجع كل بند كعنصر منفصل في القائمة.
7. إذا كانت الفقرة تحتوي على أكثر من حكم قانوني مستقل، قسّمها إلى بنود منفصلة.
8. تجاهل تمامًا: أرقام الصفحات، الترويسات، التذييلات، التوقيعات، الأختام، والعناصر الزخرفية.

أرجع النتيجة بصيغة JSON فقط، بدون أي نص إضافي، بالشكل التالي بالضبط:
{"clauses": [{"clause_id": 1, "text": "..."}, {"clause_id": 2, "text": "..."}]}

نص العقد:
\"\"\"
{contract_text}
\"\"\"
"""

def extract_clauses_with_llm(contract_text):
    """Sends raw contract text to LLM and returns list of clause dicts."""
    prompt = CLAUSE_EXTRACTION_PROMPT.replace('{contract_text}', contract_text)

    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': 'أنت أداة استخراج نصوص قانونية دقيقة. لا تجب إلا بصيغة JSON صالحة فقط.'},
            {'role': 'user',   'content': prompt}
        ],
        temperature=0.0,
        max_tokens=8000,
        response_format={'type': 'json_object'},
    )

    raw = response.choices[0].message.content.strip()
    try:
        return json.loads(raw).get('clauses', [])
    except json.JSONDecodeError as e:
        print(f'⚠️ JSON parse error: {e}')
        return []


def chunk_text(text, max_chars=12000, overlap=500):
    """Splits long contracts into overlapping chunks to stay within LLM context limits."""
    if len(text) <= max_chars:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + max_chars])
        start += max_chars - overlap
    return chunks


print('✅ Clause extraction functions ready')

✅ Clause extraction functions ready


---
## Cell 8 — Main Pipeline + API Response Formatter

In [8]:
from typing import Any, Dict

def _to_serializable(obj):
    """Recursively converts numpy/non-JSON types to plain Python for JSON serialization."""
    if isinstance(obj, dict):
        return {k: _to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_serializable(i) for i in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def format_contract_analysis_response(analysis: Dict[str, Any]) -> Dict[str, Any]:
    """Ensures analyze_contract() output exactly matches the API response schema."""
    clauses = [
        {
            'clause_id':      int(c['clause_id']),
            'clause_text':    str(c['clause_text']),
            'risk':           bool(c['risk']),
            'risk_type':      c.get('risk_type')       if c['risk'] else None,
            'risk_parties':   c.get('risk_parties', []) if c['risk'] else [],
            'explanation':    c.get('explanation')     if c['risk'] else None,
            'risk_level':     c.get('risk_level')      if c['risk'] else None,
            'recommendation': c.get('recommendation')  if c['risk'] else None,
        }
        for c in analysis.get('contract_analysis', [])
    ]
    total = len(clauses)
    risky = sum(1 for c in clauses if c['risk'])
    return _to_serializable({
        'contract_analysis': clauses,
        'summary': {
            'total_clauses': analysis.get('summary', {}).get('total_clauses', total),
            'risky_clauses': analysis.get('summary', {}).get('risky_clauses', risky),
            'safe_clauses':  analysis.get('summary', {}).get('safe_clauses',  total - risky),
        },
    })

def analyze_contract(file_path, contract_type='عام', contract_subtype='عام'):
    """
    Full pipeline:
      PDF/Image → raw text → LLM clause extraction → ML prediction → LLM explanation → API response

    Args:
        file_path:        path to PDF or image file
        contract_type:    Arabic contract category (e.g. 'عقد إيجار')
        contract_subtype: Arabic contract subtype  (e.g. 'إيجار سكني')

    Returns:
        dict matching the API response schema with contract_analysis + summary
    """
    # Step 1: Extract raw text
    print('📄 Extracting text from document...')
    raw_text = extract_text(file_path)
    print(f'   → {len(raw_text)} characters extracted')

    # Step 2: Split into clauses via LLM
    print('🔍 Extracting clauses...')
    chunks     = chunk_text(raw_text)
    all_clauses = []
    clause_counter = 1

    for i, chunk in enumerate(chunks):
        print(f'   → Processing chunk {i+1}/{len(chunks)}')
        extracted = extract_clauses_with_llm(chunk)
        for c in extracted:
            all_clauses.append({'clause_id': clause_counter, 'text': c.get('text', '').strip()})
            clause_counter += 1

    print(f'   → {len(all_clauses)} clauses extracted')

    # Step 3: Predict + explain each clause
    print('🤖 Running risk analysis...')
    analyzed = []

    for c in all_clauses:
        prediction = predict_clause(c['text'], contract_type, contract_subtype)

        explanation = None
        if prediction['risk'] == 1:
            explanation = explain_clause_risk(
                clause      = c['text'],
                parties     = prediction['risk_parties'],
                risk_type_ar= prediction['risk_type'],
            )

        explanation    = None
        risk_level     = None
        recommendation = None

        if prediction['risk'] == 1:
            llm_result     = explain_clause_risk(
                clause       = c['text'],
                parties      = prediction['risk_parties'],
                risk_type_ar = prediction['risk_type'],
            )
            explanation    = llm_result['explanation']
            risk_level     = llm_result['risk_level']
            recommendation = llm_result['recommendation']

        analyzed.append({
            'clause_id':      c['clause_id'],
            'clause_text':    c['text'],
            'risk':           prediction['risk'],
            'risk_type':      prediction['risk_type'],
            'risk_parties':   prediction['risk_parties'],
            'explanation':    explanation,
            'risk_level':     risk_level,
            'recommendation': recommendation,
        })

    # Step 4: Format into final API response schema
    raw_response = {
        'contract_analysis': analyzed,
        'summary': {
            'total_clauses': len(analyzed),
            'risky_clauses': sum(1 for c in analyzed if c['risk']),
            'safe_clauses':  sum(1 for c in analyzed if not c['risk']),
        }
    }

    return format_contract_analysis_response(raw_response)


print('✅ analyze_contract() and format_contract_analysis_response() ready')

✅ analyze_contract() and format_contract_analysis_response() ready


---
## Cell 9 — Run the Pipeline

In [ ]:
import json

# ── Update these two lines for your contract ──────────────────────────────
FILE_PATH       = 'sample_contract.pdf'   # path to your PDF or image
CONTRACT_TYPE   = 'عقد بيع'                         # Arabic contract category
CONTRACT_SUBTYPE= 'بيع عقار'                        # Arabic contract subtype
# ─────────────────────────────────────────────────────────────────────────

result = analyze_contract(FILE_PATH, CONTRACT_TYPE, CONTRACT_SUBTYPE)

# Pretty-print the full JSON response
print(json.dumps(result, ensure_ascii=False, indent=2))

📄 Extracting text from document...
   → 2968 characters extracted
🔍 Extracting clauses...
   → Processing chunk 1/1
   → 16 clauses extracted
🤖 Running risk analysis...
{
  "contract_analysis": [
    {
      "clause_id": 1,
      "clause_text": "تمهيد: يمتلك الطرف االول شقة سكنية الكائن في الجيزة المهندسين بمساحة ٠٢١ متر مربع، وذلك بموجب شهادة تسجيل بالشهر العقاري، وحدوده كما يلي: • الحد الشرقي شارع رئيسي • الحد الغربي عقار مجاور • الحد البحري حديقة • الحد القبلي شارع جانبي",
      "risk": false,
      "risk_type": null,
      "risk_parties": [],
      "explanation": null,
      "risk_level": null,
      "recommendation": null
    },
    {
      "clause_id": 2,
      "clause_text": "البند االول: يعتبر التمهيد السابق جزءاً ال يتجزا من العقد",
      "risk": false,
      "risk_type": null,
      "risk_parties": [],
      "explanation": null,
      "risk_level": null,
      "recommendation": null
    },
    {
      "clause_id": 3,
      "clause_text": "البند الثاني: باع الطرف االول للطرف ا

---
## Cell 10 — Save Output to File (optional)

In [ ]:
import json

OUTPUT_PATH = '/content/contract_analysis_result.json'

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f'✅ Result saved to {OUTPUT_PATH}')
print(f"   Total clauses: {result['summary']['total_clauses']}")
print(f"   Risky clauses: {result['summary']['risky_clauses']}")
print(f"   Safe clauses:  {result['summary']['safe_clauses']}")

---
## Cell 11 — Single Clause Test (without a file)
Use this to test a specific clause directly, without uploading a document.

In [9]:
test_clause = 'يحق للطرف الأول تنفيذ خطاب الضمان البنكي في حال إخلال الطرف الثاني بأي من التزاماته التعاقدية'

pred = predict_clause(test_clause, contract_type='عقد ضمان', contract_subtype='ضمان بنكي')

if pred['risk']:
    explanation = explain_clause_risk(test_clause, pred['risk_parties'], pred['risk_type'])
else:
    explanation = 'لا يوجد خطر متوقع في هذا البند.'

print('البند:            ', test_clause)
print('الخطر:            ', 'نعم ⚠️' if pred['risk'] else 'لا ✅')
print('الأطراف المتأثرة: ', pred['risk_parties'])
print('نوع الخطر:        ', pred['risk_type'])
print('\nالشرح:')
print(explanation)

البند:             يحق للطرف الأول تنفيذ خطاب الضمان البنكي في حال إخلال الطرف الثاني بأي من التزاماته التعاقدية
الخطر:             نعم ⚠️
الأطراف المتأثرة:  ['الطرف الأول']
نوع الخطر:         مسؤولية

الشرح:
{'explanation': 'يُعتبر هذا البند خطرًا على الطرف الثاني لأن إخلاله بأي التزاماته التعاقدية قد يؤدي إلى تنفيذ خطاب الضمان البنكي، مما قد يترتب عليه آثار مالية سلبية. يظهر أن الطرف الأول هو الذي يمتلك الحق في تنفيذ هذا الخيار، مما يعزز من سيطرته على الوضع. هذا البند قد يؤثر على استقرار الطرف الثاني المالي بسبب الخسائر المحتملة.', 'risk_level': 'مرتفع', 'recommendation': 'يجب على الطرف الثاني الانتباه بعناية إلى جميع التزاماته التعاقدية لتفادي أي مخالفات قد تؤدي إلى تنفيذ خطاب الضمان البنكي.'}
